# Personal Loan Approval Assistant — Machine Learning

This notebook builds the AI model for the **Personal Loan Approval Assistant** project.

**What this model does:**
- Looks at a loan applicant's financial info (income, debts, credit history, etc.)
- Predicts their **credit risk** (Low / Medium / High)
- Predicts whether the loan should be **Approved or Rejected**

**Tech used:** Python, Pandas, Scikit-Learn (Decision Tree & Random Forest)

## Step 1: Import the libraries we need

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


np.random.seed(42)


## Step 2: Create the dataset

In a real bank you would load a CSV of real applicants (`pd.read_csv(...)`).
Here we **generate a fake but realistic dataset** of 2000 loan applicants so the notebook runs on its own without needing an external file.

In [ ]:
n = 2000  

age = np.random.randint(18, 65, n)
monthly_income = np.random.randint(3000, 40000, n)          # EGP per month
employment_status = np.random.choice(['Employed', 'Self-Employed', 'Unemployed'], n, p=[0.6, 0.25, 0.15])
years_employed = np.random.randint(0, 30, n)

credit_history_years = np.random.randint(0, 20, n)
existing_loans = np.random.randint(0, 5, n)
late_payments = np.random.randint(0, 6, n)
defaulted_before = np.random.choice([0, 1], n, p=[0.85, 0.15])   # 1 = defaulted before

requested_amount = np.random.randint(5000, 200000, n)
monthly_debt = np.random.randint(0, 5000, n)

data = pd.DataFrame({
    'age': age,
    'monthly_income': monthly_income,
    'employment_status': employment_status,
    'years_employed': years_employed,
    'credit_history_years': credit_history_years,
    'existing_loans': existing_loans,
    'late_payments': late_payments,
    'defaulted_before': defaulted_before,
    'requested_amount': requested_amount,
    'monthly_debt': monthly_debt,
})

data.head()


,age,monthly_income,employment_status,years_employed,credit_history_years,existing_loans,late_payments,defaulted_before,requested_amount,monthly_debt
0,56,38984,Self-Employed,24,16,0,5,0,70950,4845
1,46,29616,Self-Employed,3,6,1,4,1,64612,3598
2,32,5113,Employed,9,13,2,1,0,69873,2392
3,60,16760,Unemployed,18,11,0,3,0,120792,4112
4,25,24927,Employed,17,16,4,1,0,198074,2871


## Step 3: Create the two things we want to PREDICT (our targets)

We calculate a simple **credit score** and a **risk level** using a formula (this is how the training data gets its "correct answers"). The model will later learn to guess these from the applicant info alone.

**Important:** the credit score formula includes `debt_to_income`, so applicants who ask for a lot of debt relative to their income get a lower score. This makes the model actually react when you change income, debt, or the requested amount.

In [ ]:
data['debt_to_income'] = data['monthly_debt'] / data['monthly_income']

data['loan_to_income'] = data['requested_amount'] / (data['monthly_income'] * 12)


data['credit_score'] = (
    650
    + data['credit_history_years'] * 5      # longer history = better score
    - data['late_payments'] * 20            # late payments hurt the score
    - data['defaulted_before'] * 100        # defaulting before hurts a lot
    - data['existing_loans'] * 10           # more existing loans = more risk
    - data['debt_to_income'] * 120          # higher debt-to-income = lower score
    - data['loan_to_income'] * 60           # asking for a huge loan vs income = lower score
)
data['credit_score'] = data['credit_score'].clip(300, 850)  # keep score in valid range

# --- risk_level: Low / Medium / High based on the credit score ---
def get_risk(score):
    if score >= 680:
        return 'Low'
    elif score >= 550:
        return 'Medium'
    else:
        return 'High'

data['risk_level'] = data['credit_score'].apply(get_risk)


data['approved'] = np.where(
    (data['risk_level'] == 'Low') |
    ((data['risk_level'] == 'Medium') & (data['defaulted_before'] == 0)),
    1, 0
)

data[['credit_score', 'debt_to_income', 'loan_to_income', 'risk_level', 'approved']].head()


,credit_score,debt_to_income,loan_to_income,risk_level,approved
0,605.986302,0.124282,0.151665,Medium,1
1,464.513101,0.121488,0.181805,High,0
2,550.531977,0.467827,1.138813,Medium,1
3,579.522673,0.245346,0.600597,Medium,1
4,616.448028,0.115176,0.662180,Medium,1


### Save the full dataset to a CSV file

Now that the table has all the applicant info AND the calculated answers (credit_score, risk_level, approved), we save it to `loan_applicants.csv`. This is the actual dataset file used to train the models below.

In [4]:
data.to_csv('loan_applicants.csv', index=False)
print('Saved loan_applicants.csv with', len(data), 'rows and', len(data.columns), 'columns')


Saved loan_applicants.csv with 2000 rows and 15 columns


## Step 4: Check for missing values

Before training, we always check if any data is missing (empty cells).

In [5]:
data.isnull().sum()   # shows 0 for every column because our data is clean


age                     0
monthly_income          0
employment_status       0
years_employed          0
credit_history_years    0
existing_loans          0
late_payments           0
defaulted_before        0
requested_amount        0
monthly_debt            0
debt_to_income          0
loan_to_income          0
credit_score            0
risk_level              0
approved                0
dtype: int64

## Step 5: Encode text columns into numbers

Machine learning models only understand numbers, not text like 'Employed'. So we convert the `employment_status` text column into number columns using **one-hot encoding**.

In [ ]:

data_encoded = pd.get_dummies(data, columns=['employment_status'])

data_encoded.head()


,age,monthly_income,years_employed,credit_history_years,existing_loans,late_payments,defaulted_before,requested_amount,monthly_debt,debt_to_income,loan_to_income,credit_score,risk_level,approved,employment_status_Employed,employment_status_Self-Employed,employment_status_Unemployed
0,56,38984,24,16,0,5,0,70950,4845,0.124282,0.151665,605.986302,Medium,1,False,True,False
1,46,29616,3,6,1,4,1,64612,3598,0.121488,0.181805,464.513101,High,0,False,True,False
2,32,5113,9,13,2,1,0,69873,2392,0.467827,1.138813,550.531977,Medium,1,True,False,False
3,60,16760,18,11,0,3,0,120792,4112,0.245346,0.600597,579.522673,Medium,1,False,False,True
4,25,24927,17,16,4,1,0,198074,2871,0.115176,0.662180,616.448028,Medium,1,True,False,False


## Step 6: Split the data into Features (X) and Targets (y)

- **X** = the information we know about the applicant (the inputs)
- **y_risk** = the risk level we want to predict
- **y_approved** = the approve/reject decision we want to predict

In [7]:
feature_columns = [
    'age', 'monthly_income', 'years_employed', 'credit_history_years',
    'existing_loans', 'late_payments', 'defaulted_before',
    'requested_amount', 'monthly_debt', 'credit_score', 'debt_to_income', 'loan_to_income',
    'employment_status_Employed', 'employment_status_Self-Employed', 'employment_status_Unemployed'
]

X = data_encoded[feature_columns]
y_risk = data_encoded['risk_level']
y_approved = data_encoded['approved']

# split into 80% training data, 20% testing data
X_train, X_test, y_risk_train, y_risk_test, y_appr_train, y_appr_test = train_test_split(
    X, y_risk, y_approved, test_size=0.2, random_state=42
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


Training rows: 1600
Testing rows: 400


## Step 7: Train the Risk Model (Random Forest)

A **Random Forest** builds many small decision trees and lets them vote — this usually gives more accurate and stable predictions than a single tree.

In [8]:
risk_model = RandomForestClassifier(n_estimators=100, random_state=42)
risk_model.fit(X_train, y_risk_train)   # this is where the model "learns"

# test how well it performs on data it has NEVER seen before
risk_predictions = risk_model.predict(X_test)
print('Risk Model Accuracy:', accuracy_score(y_risk_test, risk_predictions))
print()
print(classification_report(y_risk_test, risk_predictions))


Risk Model Accuracy: 1.0

              precision    recall  f1-score   support

        High       1.00      1.00      1.00       180
         Low       1.00      1.00      1.00        10
      Medium       1.00      1.00      1.00       210

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



## Step 8: Train the Approval Model (Decision Tree)

A **Decision Tree** asks a series of yes/no questions (like a flowchart) to reach a final approve/reject decision. It's simple and easy to explain in a demo.

In [9]:
approval_model = DecisionTreeClassifier(max_depth=5, random_state=42)
approval_model.fit(X_train, y_appr_train)

approval_predictions = approval_model.predict(X_test)
print('Approval Model Accuracy:', accuracy_score(y_appr_test, approval_predictions))
print()
print(confusion_matrix(y_appr_test, approval_predictions))
print()
print(classification_report(y_appr_test, approval_predictions))


Approval Model Accuracy: 1.0

[[184   0]
 [  0 216]]

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       184
           1       1.00      1.00      1.00       216

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



## Step 9: Test the model on ONE new applicant

This is exactly what will happen inside the Streamlit and Flet apps: we take one applicant's info and ask the model to predict their risk & approval.

In [10]:
# example: a new applicant applying for a loan
new_applicant = pd.DataFrame([{
    'age': 30,
    'monthly_income': 15000,
    'years_employed': 5,
    'credit_history_years': 6,
    'existing_loans': 1,
    'late_payments': 0,
    'defaulted_before': 0,
    'requested_amount': 40000,
    'monthly_debt': 1500,
    'credit_score': 700,          # you can also calculate this the same way as Step 3
    'debt_to_income': 1500/15000,
    'loan_to_income': 40000/(15000*12),
    'employment_status_Employed': 1,
    'employment_status_Self-Employed': 0,
    'employment_status_Unemployed': 0,
}])

predicted_risk = risk_model.predict(new_applicant)[0]
predicted_approval = approval_model.predict(new_applicant)[0]

print('Predicted risk level:', predicted_risk)
print('Loan approved?', 'YES' if predicted_approval == 1 else 'NO')


Predicted risk level: Low
Loan approved? YES


## Step 10: Save the trained models to files

We save both models to `.pkl` files using `pickle`, so the Streamlit and Flet apps can **load them instantly** without retraining every time.

In [11]:
import pickle

with open('risk_model.pkl', 'wb') as f:
    pickle.dump(risk_model, f)

with open('approval_model.pkl', 'wb') as f:
    pickle.dump(approval_model, f)

print('Models saved: risk_model.pkl, approval_model.pkl')


Models saved: risk_model.pkl, approval_model.pkl


### That's it! ✅

We now have two saved models (`risk_model.pkl` and `approval_model.pkl`) ready to be used by:
- `streamlit_app.py` → the Web App for applicants
- `flet_app.py` → the Desktop App for bank agents